# Run experiment with DKT on ASSISTments 2017

## Imports

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os
import random
import numpy as np
import pandas as pd
import torch

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Project paths
BASE_DIR = "/content/drive/MyDrive/education-ml-research"
DATA_DIR = os.path.join(BASE_DIR, "ASSISTments2017")
RESULTS_DIR = os.path.join(DATA_DIR, "reliability_experiment_irt")

os.makedirs(RESULTS_DIR, exist_ok=True)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

print("Data directory:", DATA_DIR)
print("Results directory:", RESULTS_DIR)
print("Results directory exists:", os.path.exists(RESULTS_DIR))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Device: cuda
GPU: NVIDIA A100-SXM4-40GB
Data directory: /content/drive/MyDrive/education-ml-research/ASSISTments2017
Results directory: /content/drive/MyDrive/education-ml-research/ASSISTments2017/reliability_experiment_irt
Results directory exists: True


## Load dataset

In [ ]:
import os
import numpy as np
import pandas as pd
import torch

# Paths
BASE_DIR = "/content/drive/MyDrive/education-ml-research"
DATA_DIR = os.path.join(BASE_DIR, "ASSISTments2017")
RESULTS_DIR = os.path.join(DATA_DIR, "reliability_experiment_irt")

RAW_PATH = os.path.join(
    DATA_DIR, "anonymized_full_release_competition_dataset.csv"
)
IRT_PATH = os.path.join(
    DATA_DIR, "student_irt_ability_groups.csv"
)

# Load data
raw_df = pd.read_csv(RAW_PATH)
irt_df = pd.read_csv(IRT_PATH)

print("Device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")
print("\nRaw dataset shape:", raw_df.shape)
print("IRT dataset shape:", irt_df.shape)

print("\nRaw columns:")
print(raw_df.columns.tolist())

print("\nIRT columns:")
print(irt_df.columns.tolist())

print("\nUnique raw students:", raw_df["studentId"].nunique())
print("Unique IRT students:", irt_df["studentId"].nunique())
print("\nIRT quartile counts:")
print(irt_df["theta_quartile"].value_counts().sort_index())

/tmp/ipykernel_9799/2997249483.py:19: DtypeWarning: Columns (76,77) have mixed types. Specify dtype option on import or set low_memory=False.
  raw_df = pd.read_csv(RAW_PATH)


Device: NVIDIA A100-SXM4-40GB

Raw dataset shape: (942816, 82)
IRT dataset shape: (1709, 5)

Raw columns:
['studentId', 'MiddleSchoolId', 'InferredGender', 'SY ASSISTments Usage', 'AveKnow', 'AveCarelessness', 'AveCorrect', 'NumActions', 'AveResBored', 'AveResEngcon', 'AveResConf', 'AveResFrust', 'AveResOfftask', 'AveResGaming', 'action_num', 'skill', 'problemId', 'problemType', 'assignmentId', 'assistmentId', 'startTime', 'endTime', 'timeTaken', 'correct', 'original', 'hint', 'hintCount', 'hintTotal', 'scaffold', 'bottomHint', 'attemptCount', 'frIsHelpRequest', 'frPast5HelpRequest', 'frPast8HelpRequest', 'stlHintUsed', 'past8BottomOut', 'totalFrPercentPastWrong', 'totalFrPastWrongCount', 'frPast5WrongCount', 'frPast8WrongCount', 'totalFrTimeOnSkill', 'timeSinceSkill', 'frWorkingInSchool', 'totalFrAttempted', 'totalFrSkillOpportunities', 'responseIsFillIn', 'responseIsChosen', 'endsWithScaffolding', 'endsWithAutoScaffolding', 'frTimeTakenOnScaffolding', 'frTotalSkillOpportunitiesScaffo

## Merge IRT groups and dataset

In [ ]:
# Standardize the interaction columns
dkt_df = raw_df[
    ["studentId", "problemId", "skill", "startTime", "correct"]
].copy()

dkt_df = dkt_df.rename(columns={
    "studentId": "user_id",
    "problemId": "problem_id",
    "skill": "skill_id",
    "startTime": "time",
})

# Normalize skill labels
dkt_df["skill_id"] = dkt_df["skill_id"].astype(str).str.strip()

# Exclude interactions without a meaningful skill identifier
before_rows = len(dkt_df)
dkt_df = dkt_df[dkt_df["skill_id"].ne("noskill")].copy()

print("Rows removed for noskill:", before_rows - len(dkt_df))

# Ensure identifiers and labels have the expected types
dkt_df["user_id"] = dkt_df["user_id"].astype(int)
dkt_df["problem_id"] = dkt_df["problem_id"].astype(int)
dkt_df["time"] = dkt_df["time"].astype(int)
dkt_df["correct"] = dkt_df["correct"].astype(int)

# Attach independently prepared IRT ability estimates and groups
irt_for_merge = irt_df[
    ["studentId", "theta_all", "theta_first",
     "theta_quartile", "theta_sd_group"]
].rename(columns={"studentId": "user_id"})

dkt_df = dkt_df.merge(
    irt_for_merge,
    on="user_id",
    how="left",
    validate="many_to_one"
)

# Sort interactions chronologically within each student
dkt_df = dkt_df.sort_values(
    ["user_id", "time", "problem_id"]
).reset_index(drop=True)

print("\nCleaned DKT data")
print("Rows:", len(dkt_df))
print("Unique students:", dkt_df["user_id"].nunique())
print("Unique problems:", dkt_df["problem_id"].nunique())
print("Unique skills:", dkt_df["skill_id"].nunique())
print("Missing IRT groups:", dkt_df["theta_quartile"].isna().sum())
print("Correctness values:", sorted(dkt_df["correct"].unique()))

print("\nStudents by IRT quartile:")
print(
    dkt_df[["user_id", "theta_quartile"]]
    .drop_duplicates()["theta_quartile"]
    .value_counts()
    .sort_index()
)

print("\nFirst five cleaned interactions:")
display(dkt_df.head())

Rows removed for noskill: 78103

Cleaned DKT data
Rows: 864713
Unique students: 1709
Unique problems: 2210
Unique skills: 101
Missing IRT groups: 0
Correctness values: [np.int64(0), np.int64(1)]

Students by IRT quartile:
theta_quartile
Q1    428
Q2    427
Q3    427
Q4    427
Name: count, dtype: int64

First five cleaned interactions:


,user_id,problem_id,skill_id,time,correct,theta_all,theta_first,theta_quartile,theta_sd_group
0,8,1118,properties-of-geometric-figures,1096470301,0,0.145408,0.659446,Q4,0 to +0.5 SD
1,8,1119,properties-of-geometric-figures,1096470350,1,0.145408,0.659446,Q4,0 to +0.5 SD
2,8,1120,sum-of-interior-angles-more-than-3-sides,1096470354,0,0.145408,0.659446,Q4,0 to +0.5 SD
3,8,1120,sum-of-interior-angles-more-than-3-sides,1096470360,0,0.145408,0.659446,Q4,0 to +0.5 SD
4,8,1121,sum-of-interior-angles-more-than-3-sides,1096470378,1,0.145408,0.659446,Q4,0 to +0.5 SD


## Create train, validation, and test splits

In [ ]:
from sklearn.model_selection import train_test_split

# One row per student for stratification
student_groups = (
    dkt_df[["user_id", "theta_quartile"]]
    .drop_duplicates()
    .sort_values("user_id")
)

all_student_ids = student_groups["user_id"].to_numpy()
all_quartiles = student_groups["theta_quartile"].to_numpy()

# 80% development pool, 20% held-out test set
trainval_ids, test_ids = train_test_split(
    all_student_ids,
    test_size=0.20,
    random_state=SEED,
    stratify=all_quartiles
)

# Stratify the training/validation split by quartile too
trainval_groups = student_groups[
    student_groups["user_id"].isin(trainval_ids)
]

train_ids, val_ids = train_test_split(
    trainval_groups["user_id"].to_numpy(),
    test_size=0.10,
    random_state=SEED,
    stratify=trainval_groups["theta_quartile"].to_numpy()
)

# Filter interaction rows
fit_df = dkt_df[dkt_df["user_id"].isin(train_ids)].copy()
val_df = dkt_df[dkt_df["user_id"].isin(val_ids)].copy()
test_df = dkt_df[dkt_df["user_id"].isin(test_ids)].copy()

# Verify no student leakage
assert set(train_ids).isdisjoint(val_ids)
assert set(train_ids).isdisjoint(test_ids)
assert set(val_ids).isdisjoint(test_ids)

print("Student split sizes")
print("Training:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

print("\nInteraction counts")
print("Training:", len(fit_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nQuartile counts by split")
for name, ids in [
    ("Training", train_ids),
    ("Validation", val_ids),
    ("Test", test_ids)
]:
    counts = (
        student_groups[student_groups["user_id"].isin(ids)]
        ["theta_quartile"]
        .value_counts()
        .sort_index()
    )
    print(f"\n{name}:\n{counts}")

print("\nStudent overlap")
print("Train ∩ validation:", len(set(train_ids) & set(val_ids)))
print("Train ∩ test:", len(set(train_ids) & set(test_ids)))
print("Validation ∩ test:", len(set(val_ids) & set(test_ids)))

Student split sizes
Training: 1230
Validation: 137
Test: 342

Interaction counts
Training: 637583
Validation: 68887
Test: 158243

Quartile counts by split

Training:
theta_quartile
Q1    308
Q2    308
Q3    307
Q4    307
Name: count, dtype: int64

Validation:
theta_quartile
Q1    34
Q2    34
Q3    35
Q4    34
Name: count, dtype: int64

Test:
theta_quartile
Q1    86
Q2    85
Q3    85
Q4    86
Name: count, dtype: int64

Student overlap
Train ∩ validation: 0
Train ∩ test: 0
Validation ∩ test: 0


In [ ]:
from sklearn.model_selection import train_test_split

# Force student 2003 into training
forced_train_id = 2003

# Start from the existing student-level split
train_ids_new = set(train_ids)
val_ids_new = set(val_ids)
test_ids_new = set(test_ids)

# Move student 2003 from test to training
test_ids_new.discard(forced_train_id)
val_ids_new.discard(forced_train_id)
train_ids_new.add(forced_train_id)

# Convert back to sorted NumPy arrays
train_ids = np.array(sorted(train_ids_new))
val_ids = np.array(sorted(val_ids_new))
test_ids = np.array(sorted(test_ids_new))

# Rebuild interaction-level datasets
fit_df = dkt_df[dkt_df["user_id"].isin(train_ids)].copy()
val_df = dkt_df[dkt_df["user_id"].isin(val_ids)].copy()
test_df = dkt_df[dkt_df["user_id"].isin(test_ids)].copy()

# Verify student separation
train_students = set(train_ids)
val_students = set(val_ids)
test_students = set(test_ids)

print("Student counts:")
print("Training:", len(train_ids))
print("Validation:", len(val_ids))
print("Test:", len(test_ids))

print("\nInteraction counts:")
print("Training:", len(fit_df))
print("Validation:", len(val_df))
print("Test:", len(test_df))

print("\nStudent 2003 in training:", forced_train_id in train_students)
print("Student 2003 in validation:", forced_train_id in val_students)
print("Student 2003 in test:", forced_train_id in test_students)

print("\nPairwise student overlaps:")
print("Train/validation:", len(train_students & val_students))
print("Train/test:", len(train_students & test_students))
print("Validation/test:", len(val_students & test_students))

# Rebuild the skill vocabulary from training data
skill_ids = sorted(fit_df["skill_id"].unique())
skill2idx = {skill: idx for idx, skill in enumerate(skill_ids)}
num_q = len(skill2idx)

unseen_val = set(val_df["skill_id"].unique()) - set(skill_ids)
unseen_test = set(test_df["skill_id"].unique()) - set(skill_ids)

print("\nTraining skill count:", num_q)
print("Unseen validation skills:", sorted(unseen_val))
print("Unseen test skills:", sorted(unseen_test))

Student counts:
Training: 1231
Validation: 137
Test: 341

Interaction counts:
Training: 638294
Validation: 68887
Test: 157532

Student 2003 in training: True
Student 2003 in validation: False
Student 2003 in test: False

Pairwise student overlaps:
Train/validation: 0
Train/test: 0
Validation/test: 0

Training skill count: 101
Unseen validation skills: []
Unseen test skills: []


In [ ]:
# Find the unseen skill and the students who have attempted it
print("Unseen test skills:", sorted(unseen_test))

unseen_rows = test_df[test_df["skill_id"].isin(unseen_test)]

display(
    unseen_rows[
        ["user_id", "problem_id", "skill_id", "time", "correct"]
    ].sort_values(["user_id", "time"])
)

unseen_students = unseen_rows["user_id"].unique()

print("\nStudents with the unseen skill:", unseen_students.tolist())

for uid in unseen_students:
    print(f"\nStudent {uid}:")
    print("Currently in test:", uid in set(test_ids))
    print("Currently in training:", uid in set(train_ids))
    print("Currently in validation:", uid in set(val_ids))

    student_skills = set(
        dkt_df.loc[dkt_df["user_id"] == uid, "skill_id"]
    )
    print("Skills unique to this student:",
          sorted(student_skills - set(fit_df["skill_id"].unique())))

Unseen test skills: []


,user_id,problem_id,skill_id,time,correct



Students with the unseen skill: []


## Build skill and student sequences

In [ ]:
# Build chronological skill-response sequences for each split

def build_sequences(df, skill2idx):
    q_seqs = []
    r_seqs = []
    student_ids = []
    theta_quartiles = []
    theta_sd_groups = []

    df = df.sort_values(["user_id", "time", "problem_id"])

    for user_id, group in df.groupby("user_id", sort=False):
        q = group["skill_id"].map(skill2idx).to_numpy(dtype=np.int64)
        r = group["correct"].to_numpy(dtype=np.float32)

        q_seqs.append(q)
        r_seqs.append(r)
        student_ids.append(int(user_id))
        theta_quartiles.append(group["theta_quartile"].iloc[0])
        theta_sd_groups.append(group["theta_sd_group"].iloc[0])

    return {
        "q_seqs": q_seqs,
        "r_seqs": r_seqs,
        "student_ids": student_ids,
        "theta_quartiles": theta_quartiles,
        "theta_sd_groups": theta_sd_groups,
    }


train_data = build_sequences(fit_df, skill2idx)
val_data = build_sequences(val_df, skill2idx)
test_data = build_sequences(test_df, skill2idx)

for name, data in [
    ("Train", train_data),
    ("Validation", val_data),
    ("Test", test_data),
]:
    lengths = np.array([len(seq) for seq in data["q_seqs"]])

    print(f"\n{name}")
    print("Students:", len(data["student_ids"]))
    print("Total interactions:", int(lengths.sum()))
    print("Minimum sequence length:", int(lengths.min()))
    print("Median sequence length:", float(np.median(lengths)))
    print("Maximum sequence length:", int(lengths.max()))
    print("Invalid skill indices:", sum(
        np.any((q < 0) | (q >= num_q)) for q in data["q_seqs"]
    ))
    print("Invalid response values:", sum(
        np.any((r != 0) & (r != 1)) for r in data["r_seqs"]
    ))

# Inspect one example training sequence
example_idx = 0
print("\nExample student:", train_data["student_ids"][example_idx])
print("First 15 skill indices:", train_data["q_seqs"][example_idx][:15])
print("First 15 responses:", train_data["r_seqs"][example_idx][:15])


Train
Students: 1231
Total interactions: 638294
Minimum sequence length: 2
Median sequence length: 400.0
Maximum sequence length: 2593
Invalid skill indices: 0
Invalid response values: 0

Validation
Students: 137
Total interactions: 68887
Minimum sequence length: 15
Median sequence length: 384.0
Maximum sequence length: 1725
Invalid skill indices: 0
Invalid response values: 0

Test
Students: 341
Total interactions: 157532
Minimum sequence length: 11
Median sequence length: 368.0
Maximum sequence length: 3057
Invalid skill indices: 0
Invalid response values: 0

Example student: 8
First 15 skill indices: [71 71 91 91 91 91 91 68 96 96 96 96 68 68 68]
First 15 responses: [0. 1. 0. 0. 1. 0. 1. 0. 0. 0. 1. 1. 0. 1. 0.]


## Chunk sequences for DKT

In [ ]:
# Chunk each student's chronological sequence into windows
# Each chunk contains seq_len + 1 interactions so we can
# use interactions[:-1] as input and interactions[1:] as targets.

SEQ_LEN = 100

def chunk_sequences(data, seq_len=SEQ_LEN):
    chunks = []

    for i, (q, r) in enumerate(zip(data["q_seqs"], data["r_seqs"])):
        student_id = data["student_ids"][i]
        theta_quartile = data["theta_quartiles"][i]
        theta_sd_group = data["theta_sd_groups"][i]

        # Non-overlapping chunks; keep the final shorter chunk if it
        # has at least two interactions to form an input-target pair.
        start = 0
        while start < len(q) - 1:
            end = min(start + seq_len + 1, len(q))

            if end - start >= 2:
                chunks.append({
                    "student_id": student_id,
                    "theta_quartile": theta_quartile,
                    "theta_sd_group": theta_sd_group,
                    "q": q[start:end],
                    "r": r[start:end],
                })

            # The final interaction of a chunk becomes the first
            # interaction of the next chunk, preserving all transitions.
            start += seq_len

    return chunks


train_chunks = chunk_sequences(train_data)
val_chunks = chunk_sequences(val_data)
test_chunks = chunk_sequences(test_data)

for name, chunks in [
    ("Train", train_chunks),
    ("Validation", val_chunks),
    ("Test", test_chunks),
]:
    lengths = np.array([len(chunk["q"]) for chunk in chunks])

    print(f"\n{name}")
    print("Chunks:", len(chunks))
    print("Total interactions represented:", int(lengths.sum()))
    print("Minimum chunk length:", int(lengths.min()))
    print("Median chunk length:", float(np.median(lengths)))
    print("Maximum chunk length:", int(lengths.max()))
    print("Chunks longer than", SEQ_LEN + 1, ":", int((lengths > SEQ_LEN + 1).sum()))

print("\nExample chunk:")
print("Student:", train_chunks[0]["student_id"])
print("Skill sequence shape:", train_chunks[0]["q"].shape)
print("Response sequence shape:", train_chunks[0]["r"].shape)
print("First 10 skills:", train_chunks[0]["q"][:10])
print("First 10 responses:", train_chunks[0]["r"][:10])


Train
Chunks: 6991
Total interactions represented: 644054
Minimum chunk length: 2
Median chunk length: 101.0
Maximum chunk length: 101
Chunks longer than 101 : 0

Validation
Chunks: 755
Total interactions represented: 69505
Minimum chunk length: 2
Median chunk length: 101.0
Maximum chunk length: 101
Chunks longer than 101 : 0

Test
Chunks: 1742
Total interactions represented: 158933
Minimum chunk length: 2
Median chunk length: 101.0
Maximum chunk length: 101
Chunks longer than 101 : 0

Example chunk:
Student: 8
Skill sequence shape: (101,)
Response sequence shape: (101,)
First 10 skills: [71 71 91 91 91 91 91 68 96 96]
First 10 responses: [0. 1. 0. 0. 1. 0. 1. 0. 0. 0.]


## Build dataloaders

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader

class DKTChunkDataset(Dataset):
    def __init__(self, chunks):
        self.chunks = chunks

    def __len__(self):
        return len(self.chunks)

    def __getitem__(self, idx):
        chunk = self.chunks[idx]
        return {
            "q": torch.tensor(chunk["q"], dtype=torch.long),
            "r": torch.tensor(chunk["r"], dtype=torch.float32),
            "student_id": int(chunk["student_id"]),
            "theta_quartile": chunk["theta_quartile"],
            "theta_sd_group": chunk["theta_sd_group"],
        }


def dkt_collate_fn(batch):
    batch_size = len(batch)
    max_len = max(item["q"].numel() for item in batch)

    q = torch.zeros((batch_size, max_len), dtype=torch.long)
    r = torch.zeros((batch_size, max_len), dtype=torch.float32)
    mask = torch.zeros((batch_size, max_len - 1), dtype=torch.bool)

    for i, item in enumerate(batch):
        n = item["q"].numel()
        q[i, :n] = item["q"]
        r[i, :n] = item["r"]

        # A target exists for each transition from interaction t to t+1.
        mask[i, :n - 1] = True

    return {
        "q": q,
        "r": r,
        "mask": mask,
        "student_id": torch.tensor(
            [item["student_id"] for item in batch], dtype=torch.long
        ),
        "theta_quartile": [item["theta_quartile"] for item in batch],
        "theta_sd_group": [item["theta_sd_group"] for item in batch],
    }


BATCH_SIZE = 128

train_dataset = DKTChunkDataset(train_chunks)
val_dataset = DKTChunkDataset(val_chunks)
test_dataset = DKTChunkDataset(test_chunks)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=dkt_collate_fn,
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=dkt_collate_fn,
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=dkt_collate_fn,
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

batch = next(iter(train_loader))
print("\nBatch shapes:")
print("q:", batch["q"].shape)
print("r:", batch["r"].shape)
print("mask:", batch["mask"].shape)
print("Student IDs:", batch["student_id"].shape)
print("Valid transitions in batch:", int(batch["mask"].sum()))
print("Expected valid transitions:", sum(
    len(train_chunks[i]["q"]) - 1
    for i in range(min(BATCH_SIZE, len(train_chunks)))
))

Train batches: 55
Validation batches: 6
Test batches: 14

Batch shapes:
q: torch.Size([128, 101])
r: torch.Size([128, 101])
mask: torch.Size([128, 100])
Student IDs: torch.Size([128])
Valid transitions in batch: 11775
Expected valid transitions: 11367


In [ ]:
def dkt_collate_fn(batch):
    batch_size = len(batch)
    lengths = torch.tensor(
        [item["q"].numel() for item in batch],
        dtype=torch.long,
    )
    max_len = int(lengths.max())

    q = torch.zeros((batch_size, max_len), dtype=torch.long)
    r = torch.zeros((batch_size, max_len), dtype=torch.float32)
    mask = torch.zeros((batch_size, max_len - 1), dtype=torch.bool)

    for i, item in enumerate(batch):
        n = item["q"].numel()
        q[i, :n] = item["q"]
        r[i, :n] = item["r"]
        mask[i, :n - 1] = True

    return {
        "q": q,
        "r": r,
        "mask": mask,
        "lengths": lengths,
        "student_id": torch.tensor(
            [item["student_id"] for item in batch], dtype=torch.long
        ),
        "theta_quartile": [item["theta_quartile"] for item in batch],
        "theta_sd_group": [item["theta_sd_group"] for item in batch],
    }


# Recreate the loaders with the corrected collate function
train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True,
    collate_fn=dkt_collate_fn
)
val_loader = DataLoader(
    val_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=dkt_collate_fn
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False,
    collate_fn=dkt_collate_fn
)

batch = next(iter(train_loader))

expected_transitions = int((batch["lengths"] - 1).sum())
actual_transitions = int(batch["mask"].sum())

print("Batch shapes:")
print("q:", batch["q"].shape)
print("r:", batch["r"].shape)
print("mask:", batch["mask"].shape)
print("lengths:", batch["lengths"].shape)

print("\nValid transitions from lengths:", expected_transitions)
print("Valid transitions from mask:", actual_transitions)
print("Counts match:", expected_transitions == actual_transitions)

assert expected_transitions == actual_transitions
print("Mask verification passed.")

Batch shapes:
q: torch.Size([128, 101])
r: torch.Size([128, 101])
mask: torch.Size([128, 100])
lengths: torch.Size([128])

Valid transitions from lengths: 11473
Valid transitions from mask: 11473
Counts match: True
Mask verification passed.


## Define the DKT model

In [ ]:
import torch
import torch.nn as nn

# Use the GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

class DKT(nn.Module):
    def __init__(self, num_q, emb_size=100, hidden_size=100, dropout=0.2):
        super().__init__()

        self.num_q = num_q
        self.embedding = nn.Embedding(2 * num_q, emb_size)
        self.lstm = nn.LSTM(
            input_size=emb_size,
            hidden_size=hidden_size,
            batch_first=True,
        )
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(hidden_size, num_q)

    def forward(self, q, r):
        # Encode each observed skill-response pair as q + num_q * r.
        x = q + self.num_q * r.long()

        x = self.embedding(x)
        h, _ = self.lstm(x)
        h = self.dropout(h)

        # Output logits for all skills at each timestep.
        logits = self.fc(h)
        return logits


EMB_SIZE = 100
HIDDEN_SIZE = 100
DROPOUT = 0.2

model = DKT(
    num_q=num_q,
    emb_size=EMB_SIZE,
    hidden_size=HIDDEN_SIZE,
    dropout=DROPOUT,
).to(device)

print("\nModel parameter count:", sum(p.numel() for p in model.parameters()))

# Shape check
q = batch["q"].to(device)
r = batch["r"].to(device)

with torch.no_grad():
    logits = model(q, r)

print("Input q shape:", tuple(q.shape))
print("Input r shape:", tuple(r.shape))
print("Output logits shape:", tuple(logits.shape))

assert logits.shape == (q.shape[0], q.shape[1], num_q)
print("Model shape check passed.")

Device: cuda

Model parameter count: 111201
Input q shape: (128, 101)
Input r shape: (128, 101)
Output logits shape: (128, 101, 101)
Model shape check passed.


## Define training and eval functions



In [ ]:
from sklearn.metrics import roc_auc_score, brier_score_loss
import torch.nn.functional as F
import copy
import random
import os
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

LEARNING_RATE = 0.001
NUM_EPOCHS = 100

optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)


def run_epoch(model, loader, optimizer=None):
    training = optimizer is not None
    model.train(training)

    total_loss = 0.0
    total_targets = 0
    all_probs = []
    all_labels = []

    for batch in loader:
        q = batch["q"].to(device)
        r = batch["r"].to(device)
        mask = batch["mask"].to(device)

        # Input is the interaction at time t.
        q_in = q[:, :-1]
        r_in = r[:, :-1]

        # Target is the response at time t+1 for its corresponding skill.
        q_target = q[:, 1:]
        r_target = r[:, 1:]

        if training:
            optimizer.zero_grad()

        with torch.set_grad_enabled(training):
            logits = model(q_in, r_in)

            # Select the logit for the skill actually attempted next.
            target_logits = logits.gather(
                dim=2, index=q_target.unsqueeze(-1)
            ).squeeze(-1)

            # Compute loss only on valid transitions.
            loss_matrix = F.binary_cross_entropy_with_logits(
                target_logits, r_target, reduction="none"
            )
            loss = loss_matrix[mask].mean()

            if training:
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
                optimizer.step()

        n_valid = int(mask.sum().item())
        total_loss += loss_matrix[mask].sum().item()
        total_targets += n_valid

        probs = torch.sigmoid(target_logits)[mask].detach().cpu().numpy()
        labels = r_target[mask].detach().cpu().numpy()

        all_probs.extend(probs.tolist())
        all_labels.extend(labels.tolist())

    all_probs = np.asarray(all_probs)
    all_labels = np.asarray(all_labels)

    auc = (
        roc_auc_score(all_labels, all_probs)
        if len(np.unique(all_labels)) == 2 else float("nan")
    )
    brier = brier_score_loss(all_labels, all_probs)

    return {
        "loss": total_loss / total_targets,
        "auc": auc,
        "brier": brier,
        "targets": total_targets,
    }


print("Training setup ready.")
print("Device:", device)
print("Learning rate:", LEARNING_RATE)
print("Epochs:", NUM_EPOCHS)

Training setup ready.
Device: cuda
Learning rate: 0.001
Epochs: 100


## Train DKT and save the best checkpoint

In [ ]:
import os
import copy
import pandas as pd
import torch
import numpy as np

os.makedirs(RESULTS_DIR, exist_ok=True)

CHECKPOINT_PATH = os.path.join(RESULTS_DIR, "dkt_assistments2017_best.pt")
HISTORY_PATH = os.path.join(RESULTS_DIR, "dkt_assistments2017_training_history.csv")

best_val_auc = -float("inf")
best_epoch = None
best_state = None
history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_metrics = run_epoch(model, train_loader, optimizer=optimizer)
    val_metrics = run_epoch(model, val_loader, optimizer=None)

    row = {
        "epoch": epoch,
        "train_loss": train_metrics["loss"],
        "train_auc": train_metrics["auc"],
        "train_brier": train_metrics["brier"],
        "train_targets": train_metrics["targets"],
        "val_loss": val_metrics["loss"],
        "val_auc": val_metrics["auc"],
        "val_brier": val_metrics["brier"],
        "val_targets": val_metrics["targets"],
    }
    history.append(row)

    improved = np.isfinite(val_metrics["auc"]) and val_metrics["auc"] > best_val_auc

    if improved:
        best_val_auc = val_metrics["auc"]
        best_epoch = epoch
        best_state = copy.deepcopy(model.state_dict())

        torch.save({
            "epoch": best_epoch,
            "model_state_dict": best_state,
            "num_q": num_q,
            "emb_size": EMB_SIZE,
            "hidden_size": HIDDEN_SIZE,
            "dropout": DROPOUT,
            "best_val_auc": best_val_auc,
            "skill2idx": skill2idx,
        }, CHECKPOINT_PATH)

    pd.DataFrame(history).to_csv(HISTORY_PATH, index=False)

    print(
        f"Epoch {epoch:03d}/{NUM_EPOCHS} | "
        f"Train loss {train_metrics['loss']:.4f} | "
        f"Train AUC {train_metrics['auc']:.4f} | "
        f"Val loss {val_metrics['loss']:.4f} | "
        f"Val AUC {val_metrics['auc']:.4f} | "
        f"Val Brier {val_metrics['brier']:.4f}"
        f"{'  <-- best' if improved else ''}"
    )

print("\nTraining complete.")
print("Best epoch:", best_epoch)
print("Best validation AUC:", best_val_auc)
print("Checkpoint:", CHECKPOINT_PATH)
print("History:", HISTORY_PATH)

Epoch 001/100 | Train loss 0.6452 | Train AUC 0.5960 | Val loss 0.6321 | Val AUC 0.6296 | Val Brier 0.2205  <-- best
Epoch 002/100 | Train loss 0.6231 | Train AUC 0.6419 | Val loss 0.6197 | Val AUC 0.6596 | Val Brier 0.2150  <-- best
Epoch 003/100 | Train loss 0.6106 | Train AUC 0.6686 | Val loss 0.6095 | Val AUC 0.6771 | Val Brier 0.2105  <-- best
Epoch 004/100 | Train loss 0.6020 | Train AUC 0.6844 | Val loss 0.6034 | Val AUC 0.6899 | Val Brier 0.2079  <-- best
Epoch 005/100 | Train loss 0.5968 | Train AUC 0.6936 | Val loss 0.6008 | Val AUC 0.6926 | Val Brier 0.2069  <-- best
Epoch 006/100 | Train loss 0.5923 | Train AUC 0.7015 | Val loss 0.5981 | Val AUC 0.6981 | Val Brier 0.2057  <-- best
Epoch 007/100 | Train loss 0.5892 | Train AUC 0.7067 | Val loss 0.5957 | Val AUC 0.7023 | Val Brier 0.2047  <-- best
Epoch 008/100 | Train loss 0.5864 | Train AUC 0.7113 | Val loss 0.5939 | Val AUC 0.7055 | Val Brier 0.2039  <-- best
Epoch 009/100 | Train loss 0.5837 | Train AUC 0.7155 | Val loss 

## Evaluate the groups

In [ ]:
import os
import torch
import numpy as np
import pandas as pd
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, brier_score_loss, log_loss

# Load the best validation-AUC checkpoint
CHECKPOINT_PATH = os.path.join(
    RESULTS_DIR, "dkt_assistments2017_best.pt"
)

checkpoint = torch.load(
    CHECKPOINT_PATH, map_location=device, weights_only=False
)
model.load_state_dict(checkpoint["model_state_dict"])
model.to(device)
model.eval()

# Collect held-out test predictions and metadata
records = []

with torch.no_grad():
    for batch in test_loader:
        q = batch["q"].to(device)
        r = batch["r"].to(device)
        mask = batch["mask"].to(device)

        q_in = q[:, :-1]
        r_in = r[:, :-1]
        q_target = q[:, 1:]
        r_target = r[:, 1:]

        logits = model(q_in, r_in)
        target_logits = logits.gather(
            2, q_target.unsqueeze(-1)
        ).squeeze(-1)

        probs = torch.sigmoid(target_logits).cpu().numpy()
        labels = r_target.cpu().numpy()
        valid = mask.cpu().numpy()

        for i in range(q.shape[0]):
            n = int(batch["lengths"][i].item()) - 1

            records.append(pd.DataFrame({
                "student_id": int(batch["student_id"][i].item()),
                "theta_quartile": [batch["theta_quartile"][i]] * n,
                "theta_sd_group": [batch["theta_sd_group"][i]] * n,
                "y_true": labels[i, :n].astype(int),
                "y_prob": probs[i, :n],
            }))

pred_df = pd.concat(records, ignore_index=True)

# Metrics helper
def calculate_metrics(group):
    y = group["y_true"].to_numpy()
    p = np.clip(group["y_prob"].to_numpy(), 1e-7, 1 - 1e-7)

    auc = roc_auc_score(y, p) if len(np.unique(y)) == 2 else np.nan

    return pd.Series({
        "students": group["student_id"].nunique(),
        "targets": len(group),
        "correct_rate": y.mean(),
        "AUC": auc,
        "Brier": brier_score_loss(y, p),
        "LogLoss": log_loss(y, p, labels=[0, 1]),
    })

# Overall test performance
overall_results = calculate_metrics(pred_df).to_frame().T
overall_results.insert(0, "model", "DKT")
overall_results.insert(0, "dataset", "ASSISTments2017")

# Performance by IRT ability quartile
quartile_results = (
    pred_df.groupby("theta_quartile", observed=True)
    .apply(calculate_metrics, include_groups=False)
    .reset_index()
)

# Performance by IRT SD group
sd_results = (
    pred_df.groupby("theta_sd_group", observed=True)
    .apply(calculate_metrics, include_groups=False)
    .reset_index()
)

# Save outputs
overall_path = os.path.join(
    RESULTS_DIR, "dkt_assistments2017_overall_test_results.csv"
)
quartile_path = os.path.join(
    RESULTS_DIR, "dkt_assistments2017_test_by_irt_quartile.csv"
)
sd_path = os.path.join(
    RESULTS_DIR, "dkt_assistments2017_test_by_irt_sd_group.csv"
)

overall_results.to_csv(overall_path, index=False)
quartile_results.to_csv(quartile_path, index=False)
sd_results.to_csv(sd_path, index=False)

print(f"Best checkpoint epoch: {checkpoint['epoch']}")
print(f"Best validation AUC: {checkpoint['best_val_auc']:.6f}")
print(f"Total test targets: {len(pred_df):,}")

print("\n=== Overall test results ===")
display(overall_results.round(6))

print("\n=== Test results by IRT ability quartile ===")
display(quartile_results.round(6))

print("\n=== Test results by IRT SD group ===")
display(sd_results.round(6))

print("\nSaved files:")
print(overall_path)
print(quartile_path)
print(sd_path)

Best checkpoint epoch: 23
Best validation AUC: 0.718062
Total test targets: 157,191

=== Overall test results ===


,dataset,model,students,targets,correct_rate,AUC,Brier,LogLoss
0,ASSISTments2017,DKT,341.0,157191.0,0.374271,0.717658,0.201685,0.588524



=== Test results by IRT ability quartile ===


,theta_quartile,students,targets,correct_rate,AUC,Brier,LogLoss
0,Q1,85.0,43374.0,0.270807,0.715766,0.176414,0.529338
1,Q2,85.0,41750.0,0.324719,0.693037,0.197613,0.579478
2,Q3,85.0,37719.0,0.402370,0.680760,0.217629,0.626361
3,Q4,86.0,34348.0,0.534296,0.700083,0.221037,0.632710



=== Test results by IRT SD group ===


,theta_sd_group,students,targets,correct_rate,AUC,Brier,LogLoss
0,+0.5 to +1.0 SD,27.0,11027.0,0.517820,0.683902,0.225929,0.643673
1,+1.0 to +1.5 SD,17.0,5532.0,0.584418,0.702400,0.217505,0.624355
2,-0.5 to 0 SD,60.0,24083.0,0.390774,0.668232,0.219200,0.630334
3,-1.0 to -0.5 SD,78.0,40588.0,0.329014,0.697613,0.197610,0.579264
4,-1.5 to -1.0 SD,66.0,32619.0,0.285478,0.707963,0.182347,0.543275
5,0 to +0.5 SD,53.0,24977.0,0.455539,0.687552,0.222622,0.637088
6,< -1.5 SD,29.0,14196.0,0.245210,0.726104,0.167581,0.509051
7,> +1.5 SD,11.0,4169.0,0.708323,0.690951,0.197052,0.577450



Saved files:
/content/drive/MyDrive/education-ml-research/ASSISTments2017/reliability_experiment_irt/dkt_assistments2017_overall_test_results.csv
/content/drive/MyDrive/education-ml-research/ASSISTments2017/reliability_experiment_irt/dkt_assistments2017_test_by_irt_quartile.csv
/content/drive/MyDrive/education-ml-research/ASSISTments2017/reliability_experiment_irt/dkt_assistments2017_test_by_irt_sd_group.csv


# DKT Test Results — ASSISTments 2017

**Best checkpoint epoch:** 23  
**Best validation AUC:** 0.718062  
**Total held-out test students:** 341  
**Total test targets:** 157,191  

## Overall Test Performance

| Dataset | Model | Best Epoch | Students | Test Targets | AUC | Brier Score | Log Loss |
|---|---|---:|---:|---:|---:|---:|---:|
| ASSISTments 2017 | DKT | 23 | 341 | 157,191 | 0.717658 | 0.201685 | 0.588524 |

## Test Performance by IRT Ability Quartile

| IRT Quartile | Students | Test Targets | Correct Rate | AUC | Brier Score | Log Loss |
|---|---:|---:|---:|---:|---:|---:|
| Q1 (Lowest Ability) | 85 | 43,374 | 0.270807 | 0.715766 | 0.176414 | 0.529338 |
| Q2 | 85 | 41,750 | 0.324719 | 0.693037 | 0.197613 | 0.579478 |
| Q3 | 85 | 37,719 | 0.402370 | 0.680760 | 0.217629 | 0.626361 |
| Q4 (Highest Ability) | 86 | 34,348 | 0.534296 | 0.700083 | 0.221037 | 0.632710 |

## Test Performance by IRT SD Group

| IRT SD Group | Students | Test Targets | Correct Rate | AUC | Brier Score | Log Loss |
|---|---:|---:|---:|---:|---:|---:|
| Below -1.5 SD | 29 | 14,196 | 0.245210 | 0.726104 | 0.167581 | 0.509051 |
| -1.5 to -1.0 SD | 66 | 32,619 | 0.285478 | 0.707963 | 0.182347 | 0.543275 |
| -1.0 to -0.5 SD | 78 | 40,588 | 0.329014 | 0.697613 | 0.197610 | 0.579264 |
| -0.5 to 0 SD | 60 | 24,083 | 0.390774 | 0.668232 | 0.219200 | 0.630334 |
| 0 to +0.5 SD | 53 | 24,977 | 0.455539 | 0.687552 | 0.222622 | 0.637088 |
| +0.5 to +1.0 SD | 27 | 11,027 | 0.517820 | 0.683902 | 0.225929 | 0.643673 |
| +1.0 to +1.5 SD | 17 | 5,532 | 0.584418 | 0.702400 | 0.217505 | 0.624355 |
| Above +1.5 SD | 11 | 4,169 | 0.708323 | 0.690951 | 0.197052 | 0.577450 |

## Preliminary Observations

- Overall held-out test AUC was 0.717658.
- AUC varied across ability quartiles, reaching its lowest value in Q3 (0.680760).
- Brier score and log loss increased from Q1 to Q4, indicating larger probabilistic prediction errors in the higher-ability quartiles.
- Across IRT SD groups, AUC ranged from 0.668232 to 0.726104.
- The extreme upper SD group contained only 11 students, so its results should be interpreted cautiously.

These findings are descriptive. Further analysis, including student-level bootstrap confidence intervals and comparisons with other knowledge tracing models, is needed before drawing stronger conclusions about differences in predictive reliability across ability groups.